# NVTA — end-to-end on private agency data (the second front door)

The repo demonstrates the SAME gated process on two MPOs:

| | ARC Atlanta (public) | NVTA subarea (private) |
|---|---|---|
| Folder | `examples/arc_atlanta/` | `nvta_run/` (this one) |
| Data | bundled — works offline | **agency-restricted, bring your own** |
| Notebook | `ARC_END_TO_END.ipynb` | this notebook |
| Pipeline | `arc_pipeline.py` | `nvta_pipeline.py` |

**If you don't have the NVTA data, every cell below prints a clear
"not configured — EXPECTED" message and continues.** That is by design: the
public ARC notebook proves the identical process end-to-end without any
private data.

What this notebook demonstrates on a converted NVTA scenario (a period folder
from the dtalite4cube workflow: shapefile + OMX ➜ GMNS):
1. **check** — file inventory, intake gate, and the NVTA convention findings
   (flat PLF vs the agency φ table, truck PCE, unwired Cube reference, sparse ids)
2. **declare** — write the `submission.yml` convention declaration
3. **prepare** — apply the declared conventions explicitly → `<dir>_qa_run/`
4. **run** — the TAPLite kernel (gated; sparse agency ids are renumbered
   *inside the kernel* and every output reports the original ids)
5. **validate** — %RMSE / R² against the Cube `I4<P>VOL` reference
6. **accessibility** — the kernel-internal accessibility call via `pytaplite`

In [ ]:
import os
import nvta_pipeline as np_

# Point at YOUR converted scenario folder (or set DTALITE_NVTA_SCENARIO).
SCENARIO = os.environ.get("DTALITE_NVTA_SCENARIO")   # e.g. r"D:/FFX134_BD/Outputs/DTALite/pm"

RUN_ASSIGNMENT = False   # the kernel run never starts unless YOU set this True
print("scenario:", SCENARIO or "(not configured -- EXPECTED; cells will skip cleanly)")

## 1 — Check: inventory, intake gate, convention findings *(never runs the kernel)*

In [ ]:
np_.stage_check(SCENARIO)

## 2 — Declare the conventions (`submission.yml`)
Writes the NVTA declaration for the detected period: per-lane hourly CAPCLASS
capacities, PLF = φ/L from the agency period-factor table, vehicle-trip demand,
per-FTYPE BPR, `<P>LIMIT` access codes. Re-run the check afterwards — the
intake gate flips from BLOCKED to READY.

In [ ]:
if SCENARIO and not os.path.exists(os.path.join(SCENARIO, "submission.yml")):
    np_.stage_declare(SCENARIO)
elif SCENARIO:
    print("submission.yml already present -- keeping it")

## 3 — Prepare: apply the declared conventions, explicitly
Copies the network and demand **verbatim** (since 2026-07 the kernel renumbers
sparse ids internally and maps every output back to the original numbering),
then applies only what the declaration says: PLF = φ/L, truck PCE = 2, the Cube
`I4<P>VOL` reference wired into `ref_volume`, solver parameters printed line by
line, route store off for a lean validation run.

In [ ]:
np_.stage_prepare(SCENARIO)
QA_RUN = (SCENARIO.rstrip("/\\") + "_qa_run") if SCENARIO else None

## 4 — Run the TAPLite kernel *(gated — set `RUN_ASSIGNMENT = True` above)*
Streams every kernel line with an elapsed clock. On sparse-id scenarios you
will see: `NOTE: sparse zone ids detected (... ) Renumbered internally ...` —
the run takes seconds, and `link_performance.csv` still shows YOUR node ids.

In [ ]:
if SCENARIO and RUN_ASSIGNMENT:
    np_.stage_run(QA_RUN)
elif SCENARIO:
    print("kernel run skipped -- set RUN_ASSIGNMENT = True in the first cell")

## 5 — Validate against the Cube reference
Reference result on the FFX134 PM subarea: **%RMSE 8.2 %, R² 0.996,
assigned/ref 1.005** (30 road links with wired reference). A JSON record
(`nvta_validation.json`) is written next to the run.

In [ ]:
if SCENARIO and RUN_ASSIGNMENT:
    np_.stage_validate(QA_RUN)
np_.summary()

## 6 — Accessibility via the package API (kernel-internal call)
`pytaplite.accessibility()` runs the kernel with zero iterations: shortest-path
trees for every origin zone, written as `od_performance.csv` (the zone-to-zone
skim) and `zone_accessibility.csv` (per-zone reach), all in original ids.
For multimodal / transit accessibility measures use the sibling `access4gmns`
package, which can re-impedance its network from this kernel's congested times.

In [ ]:
if SCENARIO and RUN_ASSIGNMENT:
    import sys
    sys.path.insert(0, os.path.abspath(".."))
    import pytaplite
    acc = pytaplite.accessibility(QA_RUN)
    print(acc)
    display(acc.to_pandas("zones").head())

## Where to go next
- The full public walkthrough: [`../examples/arc_atlanta/ARC_END_TO_END.ipynb`](../examples/arc_atlanta/ARC_END_TO_END.ipynb)
- The package API design: [`../docs/API_ARCHITECTURE_REVIEW.md`](../docs/API_ARCHITECTURE_REVIEW.md)
- Full-network NVTA runs (6-mode PM, CUBE `I4PM*` checks): [`README.md`](README.md)
- Super-zones for large runs: `pytaplite.superzone(...)` + `examples/arc_atlanta/SUPERZONE.md`